# O-Level Mathematics Score Prediction — Exploratory Data Analysis

This notebook performs exploratory data analysis (EDA) on the O-Level Mathematics dataset.

**Objective:** Understand the data, identify missing values and patterns, examine relationships with the target variable `final_test`, and generate observations that can support the regression modelling stage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

DATA_PATH = "./data/O_Level_Math_Result_Prediction.csv"

df = pd.read_csv(DATA_PATH)
df.head()

## 1. Dataset Overview

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nFirst 5 records:")
display(df.head())

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

print("\nSummary statistics:")
display(df.describe(include="all").T)

### Observation

The dataset contains student-level demographic, academic, attendance, lifestyle and background information. The target variable is `final_test`, representing the student's Mathematics examination score.

`index` and `student_id` are identifiers rather than meaningful academic predictors, so they should not be used as modelling features.

## 2. Missing Values

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

missing_table = pd.DataFrame({
    "Missing Count": missing,
    "Missing Percentage": missing_pct
})

display(missing_table[missing_table["Missing Count"] > 0])

In [ ]:
missing_plot = missing[missing > 0].sort_values(ascending=True)

plt.figure(figsize=(10, 5))
missing_plot.plot(kind="barh")
plt.title("Missing Values by Feature")
plt.xlabel("Number of Missing Values")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

### Observation

Several variables contain missing values. Missing values should be handled inside the modelling pipeline rather than deleting large numbers of observations. Numerical variables can be imputed using a median, while categorical variables can be imputed using the most frequent category.

## 3. Target Variable: Mathematics Score

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df["final_test"].dropna(), bins=20, kde=True)
plt.title("Distribution of O-Level Mathematics Scores")
plt.xlabel("Final Test Score")
plt.ylabel("Number of Students")
plt.tight_layout()
plt.show()

In [ ]:
print("Target statistics:")
display(df["final_test"].describe())

print("Skewness:", round(df["final_test"].skew(), 3))
print("Median:", df["final_test"].median())
print("Mean:", df["final_test"].mean())

### Observation

The target distribution should be checked for concentration, skewness and unusual values. Understanding the score distribution is important because the regression models are designed to predict a continuous numerical outcome.

## 4. Numerical Feature Relationships

In [ ]:
numeric_features = [
    "number_of_siblings",
    "n_male",
    "n_female",
    "age",
    "hours_per_week",
    "attendance_rate",
    "final_test"
]

display(df[numeric_features].corr().round(2))

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(
    df[numeric_features].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 14))

features = [
    "hours_per_week",
    "attendance_rate",
    "age",
    "number_of_siblings",
    "n_male",
    "n_female"
]

for ax, feature in zip(axes.ravel(), features):
    sns.scatterplot(data=df, x=feature, y="final_test", alpha=0.35, ax=ax)
    ax.set_title(f"{feature} vs Final Test Score")

plt.tight_layout()
plt.show()

### Observation

The correlation matrix and scatter plots help identify variables that may have useful relationships with Mathematics performance. In particular, `hours_per_week` and `attendance_rate` are important candidates to investigate because study time and attendance can reasonably influence academic performance.

Correlation does not imply causation, so these relationships should be interpreted as predictive associations rather than proof of cause and effect.

## 5. Categorical Features

In [ ]:
categorical_features = [
    "direct_admission",
    "CCA",
    "learning_style",
    "gender",
    "tuition",
    "mode_of_transport",
    "bag_color"
]

for feature in categorical_features:
    print(f"\n{feature}")
    display(df[feature].value_counts(dropna=False).to_frame("count").head(15))

In [ ]:
# Compare average Mathematics score across selected categorical variables

selected_categories = [
    "gender",
    "tuition",
    "learning_style",
    "direct_admission",
    "mode_of_transport"
]

for feature in selected_categories:
    summary = (
        df.groupby(feature, dropna=False)["final_test"]
        .agg(["count", "mean", "median"])
        .sort_values("mean", ascending=False)
    )

    print(f"\n{feature}")
    display(summary)

In [ ]:
for feature in selected_categories:
    plt.figure(figsize=(9, 5))
    sns.boxplot(data=df, x=feature, y="final_test")
    plt.title(f"Mathematics Score by {feature}")
    plt.xlabel(feature)
    plt.ylabel("Final Test Score")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

### Observation

Categorical variables show whether different student groups have different score distributions. These differences can be useful for prediction, but categorical variables need to be encoded before they can be used by most regression algorithms.

The modelling pipeline therefore uses one-hot encoding for nominal categorical features.

## 6. Sleep and Wake Time Feature

In [ ]:
def time_to_minutes(value):
    if pd.isna(value):
        return np.nan

    try:
        hour, minute = map(int, str(value).split(":"))
        return hour * 60 + minute
    except (ValueError, TypeError):
        return np.nan


sleep_minutes = df["sleep_time"].apply(time_to_minutes)
wake_minutes = df["wake_time"].apply(time_to_minutes)

sleep_duration = wake_minutes - sleep_minutes
sleep_duration = sleep_duration.where(sleep_duration >= 0, sleep_duration + 24 * 60)

df["sleep_duration_hours"] = sleep_duration / 60

display(
    df[["sleep_time", "wake_time", "sleep_duration_hours"]].head(10)
)

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df["sleep_duration_hours"].dropna(), bins=20, kde=True)
plt.title("Distribution of Estimated Sleep Duration")
plt.xlabel("Sleep Duration (hours)")
plt.ylabel("Number of Students")
plt.tight_layout()
plt.show()

print("Correlation with final_test:",
      round(df[["sleep_duration_hours", "final_test"]].corr().iloc[0, 1], 3))

### Observation

`Sleep_duration_hours` is derived from `sleep_time` and `wake_time`. Using a numerical duration is more meaningful for regression than treating clock-time strings as ordinary categorical variables.

This engineered feature can therefore be included in the numerical preprocessing pipeline.

## 7. Potential Outliers

In [ ]:
box_features = [
    "number_of_siblings",
    "n_male",
    "n_female",
    "age",
    "hours_per_week",
    "attendance_rate",
    "final_test"
]

for feature in box_features:
    plt.figure(figsize=(9, 3))
    sns.boxplot(x=df[feature])
    plt.title(f"Boxplot: {feature}")
    plt.tight_layout()
    plt.show()

### Observation

Boxplots help identify unusually high or low observations. Extreme observations should not automatically be removed because they may represent genuine students with unusual circumstances. The modelling approach should therefore be robust to such observations where possible.

## 8. Features Selected for Modelling

In [ ]:
numerical_features = [
    "number_of_siblings",
    "n_male",
    "n_female",
    "age",
    "hours_per_week",
    "attendance_rate",
    "sleep_duration_hours"
]

nominal_features = [
    "direct_admission",
    "CCA",
    "learning_style",
    "gender",
    "tuition",
    "mode_of_transport",
    "bag_color"
]

excluded_features = ["index", "student_id"]

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(nominal_features)

print("\nExcluded identifier fields:")
print(excluded_features)

## 9. EDA Summary

The EDA supports the following modelling decisions:

1. `final_test` is a continuous numerical target, so regression is appropriate.
2. `index` and `student_id` are identifiers and should be excluded from prediction.
3. Numerical variables require missing-value handling and scaling for linear models.
4. Categorical variables require one-hot encoding.
5. `sleep_time` and `wake_time` can be transformed into the more meaningful `sleep_duration_hours` feature.
6. The dataset contains both linear and potentially non-linear relationships, supporting the evaluation of multiple regression models.
7. Linear Regression, Ridge Regression and Random Forest Regression provide a suitable combination of baseline, regularised and non-linear models.

The EDA findings are used as inputs to the modelling pipeline in `src/data_preparation.py` and `src/model_training.py`.